In [38]:
import pandas as pd

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline

In [12]:
train = pd.read_csv("data/train.csv", index_col = "id")
test = pd.read_csv("data/test.csv", index_col = "id")
submission = pd.read_csv("data/sample_submission.csv", index_col = "id")

## 1. Exploratory Data Analysis

In [31]:
def resumtable(df):
    print(f"data shape: {df.shape}")
    summary = pd.DataFrame(df.dtypes, columns = ['data type'])
    summary = summary.reset_index()
    summary = summary.rename(columns = {"index": "feature"})
    summary["num_NULL"] = df.isnull().sum().values
    summary["num_UNI"] = df.nunique().values
    summary["first_val"] = df.loc[0].values
    summary["second_val"] = df.loc[1].values
    summary["third_val"] = df.loc[2].values
    
    return summary

In [33]:
resumtable(train)

data shape: (300000, 24)


,feature,data type,num_NULL,num_UNI,first_val,second_val,third_val
0,bin_0,int64,0,2,0,0,0
1,bin_1,int64,0,2,0,1,0
2,bin_2,int64,0,2,0,0,0
3,bin_3,object,0,2,T,T,F
4,bin_4,object,0,2,Y,Y,Y
5,nom_0,object,0,3,Green,Green,Blue
6,nom_1,object,0,6,Triangle,Trapezoid,Trapezoid
7,nom_2,object,0,6,Snake,Hamster,Lion
8,nom_3,object,0,6,Finland,Russia,Russia
9,nom_4,object,0,4,Bassoon,Piano,Theremin


1. binary feature: bin_0 ~ bin_4
2. nominal feature: nom_0 ~ nom_9
3. ordinal feature: ord_0 ~ ord_5
4. others: day, month, target

binary feature:
- No Null data
- 2 unique values (because these are binary features)
- Need to change T, F, Y, N object types to int types (need dummy encoding)

Nominal feature:
- No Null data
- From nom_0 to nom_4 have at most 6 unique values, but from nom_5 to nom_9 have a lot of unique values
- Also, from nom_5 to nom_9 represent Uninterpretable values

Ordianl feature: 
- No Null data
- Except for ord_0, all the features are object types
- To recognize the order of these features, we need to print out their unique values (Because the impact on the target value differs depending on the order)

In [35]:
for i in range(3):
    feature = "ord_" + str(i)
    print(f"{feature}'s unique value: {train[feature].unique()}")

ord_0's unique value: [2 1 3]
ord_1's unique value: ['Grandmaster' 'Expert' 'Novice' 'Contributor' 'Master']
ord_2's unique value: ['Cold' 'Hot' 'Lava Hot' 'Boiling Hot' 'Freezing' 'Warm']


- The unique values of the ord_0 feature are all plain numbers, so we need to arrange them in the numerical order
- The ord_1 feature represents kaggle's ranking tiers (Novice, Contributor, Expert, Master, Grandmaster)
- The ord_2 feature represents levels of temperature, indicating how hot or cold something is

In [36]:
for i in range(3, 6):
    feature = "ord_" + str(i)
    print(f"{feature}'s unique value: {train[feature].unique()}")

ord_3's unique value: ['h' 'a' 'i' 'j' 'g' 'e' 'd' 'b' 'k' 'f' 'l' 'n' 'o' 'c' 'm']
ord_4's unique value: ['D' 'A' 'R' 'E' 'P' 'K' 'V' 'Q' 'Z' 'L' 'F' 'T' 'U' 'S' 'Y' 'B' 'H' 'J'
 'N' 'G' 'W' 'I' 'O' 'C' 'X' 'M']
ord_5's unique value: ['kr' 'bF' 'Jc' 'kW' 'qP' 'PZ' 'wy' 'Ed' 'qo' 'CZ' 'qX' 'su' 'dP' 'aP'
 'MV' 'oC' 'RL' 'fh' 'gJ' 'Hj' 'TR' 'CL' 'Sc' 'eQ' 'kC' 'qK' 'dh' 'gM'
 'Jf' 'fO' 'Eg' 'KZ' 'Vx' 'Fo' 'sV' 'eb' 'YC' 'RG' 'Ye' 'qA' 'lL' 'Qh'
 'Bd' 'be' 'hT' 'lF' 'nX' 'kK' 'av' 'uS' 'Jt' 'PA' 'Er' 'Qb' 'od' 'ut'
 'Dx' 'Xi' 'on' 'Dc' 'sD' 'rZ' 'Uu' 'sn' 'yc' 'Gb' 'Kq' 'dQ' 'hp' 'kL'
 'je' 'CU' 'Fd' 'PQ' 'Bn' 'ex' 'hh' 'ac' 'rp' 'dE' 'oG' 'oK' 'cp' 'mm'
 'vK' 'ek' 'dO' 'XI' 'CM' 'Vf' 'aO' 'qv' 'jp' 'Zq' 'Qo' 'DN' 'TZ' 'ke'
 'cG' 'tP' 'ud' 'tv' 'aM' 'xy' 'lx' 'To' 'uy' 'ZS' 'vy' 'ZR' 'AP' 'GJ'
 'Wv' 'ri' 'qw' 'Xh' 'FI' 'nh' 'KR' 'dB' 'BE' 'Bb' 'mc' 'MC' 'tM' 'NV'
 'ih' 'IK' 'Ob' 'RP' 'dN' 'us' 'dZ' 'yN' 'Nf' 'QM' 'jV' 'sY' 'wu' 'SB'
 'UO' 'Mx' 'JX' 'Ry' 'Uk' 'uJ' 'LE' 'ps' 'kE' 'MO' 'kw'

- The ord_3, ord_4, and ord_5 features are in alphabetical order
- I will encode them in alphabetical order later

In [37]:
print("day's unique val", train['day'].unique())
print("month's uniuqe val", train['month'].unique())
print("target's unique val", train['target'].unique())

day's unique val [2 7 5 4 3 1 6]
month's uniuqe val [ 2  8  1  4 10  3  7  9 12 11  5  6]
target's unique val [0 1]


- The day feature has 7 unique values, which seem to represent each day of the week (from Monday to Sunday).
- The month feature has 12 unique values, which seem to represent each month (from January to December).
- The target feature has two unique values.

### 1.1 Data visualization